In [5]:
## Dataset for Monday-WorkingHours.pcap_ISCX.csv

# --------------------------------------------------
# LOAD DATASET
# --------------------------------------------------

# Works with datasets and tables
import pandas as pd

# Works with numbers, arrays, and mathematical operations
import numpy as np

# Location of one CIC-IDS2017 dataset file
file_path = "../data/raw/MachineLearningCVE/Monday-WorkingHours.pcap_ISCX.csv"

# df = DataFrame
df = pd.read_csv(file_path)

# Displays the first 5 rows
df.head()

# --------------------------------------------------
# INITIAL INSPECTION
# --------------------------------------------------

# Shows the dataset dimensions
print("Rows and columns:", df.shape)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

# Remove extra spaces from column names
df.columns = df.columns.str.strip()

# Display all column names
df.columns.tolist()

# -----------------------------------------------
# DATA CLEANING
# -----------------------------------------------

# Copy to preserve the original loaded DataFrame
clean_df = df.copy()

# Record the number of rows before cleaning
rows_before = clean_df.shape[0]

# Replace positive/negative infinity with NaN
clean_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Count missing values before removing them
missing_before = clean_df.isnull().sum().sum()

# Count rows containing at least one missing value
rows_with_missing = clean_df.isnull().any(axis=1).sum()

# Remove rows containing NaN values
clean_df.dropna(inplace=True)

# Count duplicate rows before removing them
duplicates_before = clean_df.duplicated().sum()

# Remove duplicate rows
clean_df.drop_duplicates(inplace=True)

# Reset row indexes after deleting rows
clean_df.reset_index(drop=True, inplace=True)

# Record the number of rows after cleaning
rows_after = clean_df.shape[0]

# Display cleaning results
print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", rows_after)
print("Rows removed:", rows_before - rows_after)

print("Missing values found:", missing_before)
print("Duplicate rows found:", duplicates_before)
print("Rows with missing values:", rows_with_missing)

percent_removed = ((rows_before - rows_after) / rows_before) * 100
print("Percent of rows removed:", round(percent_removed, 2), "%")

# --------------------------------------------------
# VERIFY CLEANING
# --------------------------------------------------

print("\nCleaned dataset shape:", clean_df.shape)
print("Missing values:", clean_df.isnull().sum().sum())


numeric_df = clean_df.select_dtypes(include=[np.number])

print("Infinite values:", np.isinf(numeric_df).sum().sum())
print("Duplicate rows:", clean_df.duplicated().sum())


# Clean whitespace from traffic labels
clean_df["Label"] = clean_df["Label"].str.strip()

print("\nTraffic labels:")
print(clean_df["Label"].value_counts())

print("\nUnique labels:")
print(clean_df["Label"].unique())


clean_df.head(10)


# -----------------------------------------------
# DATA TYPE VALIDATION
# -----------------------------------------------

print("\nData Types: ")

# Verify if numeric values
print(clean_df.dtypes.value_counts())

# Identify any non-numeric columns
non_numeric_columns = clean_df.select_dtypes(exclude=[np.number]).columns


print("\nNon-numeric columns:")
print(non_numeric_columns.tolist())

feature_df = clean_df.drop(columns=["Label"])

constant_columns = feature_df.columns[
    feature_df.nunique() <= 1
]

print("\nConstant feature columns:")
print(constant_columns.tolist())
print("Number of constant features:", len(constant_columns))

# -----------------------------------------------
# FEATURE VALIDATION
# -----------------------------------------------

print("\nNumber of input features:", feature_df.shape[1])

numeric_features = feature_df.select_dtypes(include=[np.number])

negative_counts = (numeric_features < 0).sum()
negative_counts = negative_counts[negative_counts > 0]

print("\nFeatures containing negative values:")

if negative_counts.empty:
    print("No negative values found.")
else:
    print(negative_counts)

# -----------------------------------------------
# NEGATIVE FLOW DURATION INVESTIGATION
# -----------------------------------------------

negative_duration = clean_df[
    clean_df["Flow Duration"] < 0
]

print("Rows with negative Flow Duration:", len(negative_duration))

print("\nNegative Flow Duration values:")
print(negative_duration["Flow Duration"].value_counts())

display(negative_duration.T)

timing_columns = [
    "Flow Duration",
    "Flow IAT Mean",
    "Flow IAT Std",
    "Flow IAT Max",
    "Flow IAT Min",
    "Fwd IAT Total",
    "Fwd IAT Mean",
    "Bwd IAT Total",
    "Bwd IAT Mean",
    "Active Mean",
    "Idle Mean",
    "Label"
]

print("\nRelated timing values for negative Flow Duration rows:")
display(negative_duration[timing_columns])

# -----------------------------------------------
# FEATURE STATISTICS
# -----------------------------------------------

feature_stats = numeric_features.describe().T

display(feature_stats)


# -----------------------------------------------
# DATASET SUMMARY
# -----------------------------------------------

print("----- MONDAY DATASET SUMMARY -----")
print("Original rows:", rows_before)
print("Cleaned rows:", rows_after)
print("Rows removed:", rows_before - rows_after)
print("Percent removed:", round(percent_removed, 2), "%")
print("Rows with missing values:", rows_with_missing)

print("\nInput features:", feature_df.shape[1])
print("Target column: Label")

print("\nUnique labels:")
print(clean_df["Label"].unique())

print("\nConstant features:", len(constant_columns))
print("Features with negative values:", len(negative_counts))

print("\nFinal data quality:")
print("Missing values:", clean_df.isnull().sum().sum())
print("Infinite values:", np.isinf(numeric_df).sum().sum())
print("Duplicate rows:", clean_df.duplicated().sum())


Rows and columns: (529918, 79)
Number of rows: 529918
Number of columns: 79
Rows before cleaning: 529918
Rows after cleaning: 502650
Rows removed: 27268
Missing values found: 874
Duplicate rows found: 26831
Rows with missing values: 437
Percent of rows removed: 5.15 %

Cleaned dataset shape: (502650, 79)
Missing values: 0
Infinite values: 0
Duplicate rows: 0

Traffic labels:
Label
BENIGN    502650
Name: count, dtype: int64

Unique labels:
<StringArray>
['BENIGN']
Length: 1, dtype: str

Data Types: 
int64      54
float64    24
str         1
Name: count, dtype: int64

Non-numeric columns:
['Label']

Constant feature columns:
['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Number of constant features: 10

Number of input features: 78

Features containing negative values:
Flow Duration                  15
Flow Bytes/s                   

,13652,41278,76838,78296,109506,128165,143068,148908,244298,275665,279539,303013,338282,352018,432985
Destination Port,443,443,443,443,443,443,80,443,80,80,443,80,55092,443,443
Flow Duration,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
Total Fwd Packets,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
Total Backward Packets,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
Total Length of Fwd Packets,6,6,6,0,0,6,6,0,6,6,0,6,6,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Idle Mean,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Idle Std,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Idle Max,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Idle Min,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0



Related timing values for negative Flow Duration rows:


,Flow Duration,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Bwd IAT Total,Bwd IAT Mean,Active Mean,Idle Mean,Label
13652,-1,-1.0,0.0,-1,-1,0,0.0,0,0.0,0.0,0.0,BENIGN
41278,-1,-1.0,0.0,-1,-1,0,0.0,0,0.0,0.0,0.0,BENIGN
76838,-1,-1.0,0.0,-1,-1,0,0.0,0,0.0,0.0,0.0,BENIGN
78296,-1,-1.0,0.0,-1,-1,0,0.0,0,0.0,0.0,0.0,BENIGN
109506,-1,-1.0,0.0,-1,-1,0,0.0,0,0.0,0.0,0.0,BENIGN
128165,-1,-1.0,0.0,-1,-1,0,0.0,0,0.0,0.0,0.0,BENIGN
143068,-1,-1.0,0.0,-1,-1,0,0.0,0,0.0,0.0,0.0,BENIGN
148908,-1,-1.0,0.0,-1,-1,0,0.0,0,0.0,0.0,0.0,BENIGN
244298,-1,-1.0,0.0,-1,-1,0,0.0,0,0.0,0.0,0.0,BENIGN
275665,-1,-1.0,0.0,-1,-1,0,0.0,0,0.0,0.0,0.0,BENIGN


,count,mean,std,min,25%,50%,75%,max
Destination Port,502650.0,1.117500e+04,2.179433e+04,0.0,53.0,80.0,443.0,6.553500e+04
Flow Duration,502650.0,1.092655e+07,2.940021e+07,-1.0,195.0,36881.0,486588.0,1.200000e+08
Total Fwd Packets,502650.0,1.083513e+01,9.162970e+02,1.0,2.0,2.0,5.0,2.197590e+05
Total Backward Packets,502650.0,1.208788e+01,1.204721e+03,0.0,1.0,2.0,4.0,2.919220e+05
Total Length of Fwd Packets,502650.0,5.575106e+02,6.394182e+03,0.0,29.0,68.0,333.0,1.323378e+06
...,...,...,...,...,...,...,...,...
Active Min,502650.0,4.617997e+04,5.126268e+05,0.0,0.0,0.0,0.0,1.016597e+08
Idle Mean,502650.0,3.650938e+06,1.329004e+07,0.0,0.0,0.0,0.0,1.199997e+08
Idle Std,502650.0,2.134229e+05,2.227709e+06,0.0,0.0,0.0,0.0,7.514502e+07
Idle Max,502650.0,3.816180e+06,1.373624e+07,0.0,0.0,0.0,0.0,1.199997e+08


----- MONDAY DATASET SUMMARY -----
Original rows: 529918
Cleaned rows: 502650
Rows removed: 27268
Percent removed: 5.15 %
Rows with missing values: 437

Input features: 78
Target column: Label

Unique labels:
<StringArray>
['BENIGN']
Length: 1, dtype: str

Constant features: 10
Features with negative values: 12

Final data quality:
Missing values: 0
Infinite values: 0
Duplicate rows: 0
